In [1]:
import numpy as np
import pandas as pd
import Script.utilities as utilities
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
import numpy as np
from xgboost import XGBRegressor

In [2]:
### Reading the preprocess data from disk 
### Train data 
train_set=pd.read_csv('final_data/final_unique_train.csv')

### test data 
test_set=pd.read_csv('final_data/final_unique_test.csv')

smiles_train = train_set["smiles_canon"].astype(str).tolist()
smiles_test  = test_set["smiles_canon"].astype(str).tolist()
y_train = train_set["LogS"].astype(np.float32).values
y_test  = test_set["LogS"].astype(np.float32).values

In [20]:


def feats_unigram_bigram_trigram(
    smi,
    n_uni=2048, n_bi=4096, n_tri=4096,
    radius=2, use_chirality=True, use_features=False, normalize=True
):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(n_uni + n_bi + n_tri, dtype=np.float32)

    # Unigrams = Morgan counts
    uni_sv = AllChem.GetHashedMorganFingerprint(
        mol, radius=radius, nBits=n_uni,
        useChirality=use_chirality, useFeatures=use_features
    )
    uni = np.zeros(n_uni, dtype=np.float32)
    for k, v in uni_sv.GetNonzeroElements().items():
        uni[k] = float(v)

    # Bigrams = Atom Pairs
    ap = rdMolDescriptors.GetHashedAtomPairFingerprint(mol, nBits=n_bi)
    bi = np.zeros(n_bi, dtype=np.float32)
    for k, v in ap.GetNonzeroElements().items():
        bi[k % n_bi] = float(v)

    # Trigrams (4-gram path) = Topological Torsions
    tt = rdMolDescriptors.GetHashedTopologicalTorsionFingerprint(mol, nBits=n_tri)
    tri = np.zeros(n_tri, dtype=np.float32)
    for k, v in tt.GetNonzeroElements().items():
        tri[k % n_tri] = float(v)

    x = np.concatenate([uni, bi, tri]).astype(np.float32)
    if normalize:
        n = np.linalg.norm(x)
        if n > 0: x /= n
    return  uni, bi, tri, x


In [21]:

def featurize_smiles(smiles, **kwargs):
    uni, bi,tri, all = [],[],[],[]
    print(len(smiles))
    for s in smiles:
        a,b,c,d = feats_unigram_bigram_trigram(s, **kwargs)
        uni.append(a)
        bi.append(b)
        tri.append(c)
        all.append(d)
    # X = [feats_unigram_bigram_trigram(s, **kwargs) for s in smiles]
    return np.vstack(uni), np.vstack(bi), np.vstack(tri), np.vstack(all)

17937


[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors
[11:54:07] WARNING: not removing hydrogen atom without neighbors


1282


In [23]:
print(X_train_u.shape)
print(len(smiles_train))
print(len(y_train))
print(X_train_a.shape)

(17937, 2048)
17937
17937
(17937, 10240)


In [29]:
def ngram_reg(X_train, X_test, y_train, y_test, name):
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=42,
        n_jobs=0
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    n_gram=utilities.get_errors1(y_test,pred,name)
    n_gram['Descriptors_Detail']=name
    print(n_gram)

In [25]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=0
)
model.fit(X_train_b, y_train)
pred = model.predict(X_test_b)
n_gram_b=utilities.get_errors1(y_test,pred,"Bigrapm")
n_gram_b['Descriptors_Detail']='Bigrapm'
print(n_gram_b)

  Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0    Bigrapm  0.6005  0.5942  0.7708  0.8575            Bigrapm


In [26]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=0
)
model.fit(X_train_t, y_train)
pred = model.predict(X_test_t)
n_gram_t=utilities.get_errors1(y_test,pred,"Bigrapm")
n_gram_t['Descriptors_Detail']='Bigrapm'
print(n_gram_t)

  Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0    Bigrapm  0.7295  0.8622  0.9286  0.7933            Bigrapm


In [27]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=0
)
model.fit(X_train_a, y_train)
pred = model.predict(X_test_a)
n_gram_a=utilities.get_errors1(y_test,pred,"Bigrapm")
n_gram_a['Descriptors_Detail']='Bigrapm'
print(n_gram_a)

  Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0    Bigrapm  0.5762  0.5594  0.7479  0.8659            Bigrapm


In [31]:
for r in range (1,10,1):
    feat_kwargs = dict(
        n_uni=2048,   # try 1024..4096
        n_bi=4096,    # try 2048..16384
        n_tri=4096,   # set 0 to disable torsions
        radius=r,
    )
    X_train_u,X_train_b, X_train_t, X_train_a = featurize_smiles(smiles_train, **feat_kwargs)
    X_test_u,X_test_b, X_test_t, X_test_a  = featurize_smiles(smiles_test, **feat_kwargs)
    print("Radius ", r)
    ngram_reg(X_train=X_train_u, X_test=X_test_u, y_train=y_train, y_test=y_test, name = f"Unigram r={r}")
    ngram_reg(X_train=X_train_b, X_test=X_test_b, y_train=y_train, y_test=y_test, name = f"Bigram r={r}")
    ngram_reg(X_train=X_train_t, X_test=X_test_t, y_train=y_train, y_test=y_test, name = f"Trigram r={r}")
    ngram_reg(X_train=X_train_a, X_test=X_test_a, y_train=y_train, y_test=y_test, name = f"All together r={r}")

17937


[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors
[12:17:47] WARNING: not removing hydrogen atom without neighbors


1282
Radius  1
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Unigram r=1  0.5888  0.5812  0.7624  0.8606        Unigram r=1
   Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Bigram r=1  0.6005  0.5942  0.7708  0.8575         Bigram r=1
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Trigram r=1  0.7295  0.8622  0.9286  0.7933        Trigram r=1
         Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  All together r=1  0.5763  0.5652  0.7518  0.8645   All together r=1
17937


[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors
[12:33:31] WARNING: not removing hydrogen atom without neighbors


1282
Radius  2
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Unigram r=2  0.5987  0.5981  0.7734  0.8566        Unigram r=2
   Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Bigram r=2  0.6005  0.5942  0.7708  0.8575         Bigram r=2
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Trigram r=2  0.7295  0.8622  0.9286  0.7933        Trigram r=2
         Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  All together r=2  0.5762  0.5594  0.7479  0.8659   All together r=2
17937


...

1282
Radius  7
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Unigram r=7  0.6096  0.6234  0.7896  0.8505        Unigram r=7
   Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Bigram r=7  0.6005  0.5942  0.7708  0.8575         Bigram r=7
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Trigram r=7  0.7295  0.8622  0.9286  0.7933        Trigram r=7
         Model_Name    MAE     MSE    RMSE     R2 Descriptors_Detail
0  All together r=7  0.583  0.5797  0.7614  0.861   All together r=7
17937


[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:13] WARNING: not removing hydrogen atom without neighbors
[14:38:13] WARNING: not removing hydrogen atom without neighbors
[14:38:13] WARNING: not removing hydrogen atom without neighbors


1282
Radius  8
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Unigram r=8  0.6109  0.6233  0.7895  0.8505        Unigram r=8
   Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Bigram r=8  0.6005  0.5942  0.7708  0.8575         Bigram r=8
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Trigram r=8  0.7295  0.8622  0.9286  0.7933        Trigram r=8
         Model_Name     MAE    MSE    RMSE      R2 Descriptors_Detail
0  All together r=8  0.5867  0.578  0.7603  0.8614   All together r=8
17937


[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:44] WARNING: not removing hydrogen atom without neighbors
[14:57:45] WARNING: not removing hydrogen atom without neighbors
[14:57:45] WARNING: not removing hydrogen atom without neighbors
[14:57:45] WARNING: not removing hydrogen atom without neighbors


1282
Radius  9
    Model_Name     MAE    MSE    RMSE      R2 Descriptors_Detail
0  Unigram r=9  0.6099  0.623  0.7893  0.8506        Unigram r=9
   Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Bigram r=9  0.6005  0.5942  0.7708  0.8575         Bigram r=9
    Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  Trigram r=9  0.7295  0.8622  0.9286  0.7933        Trigram r=9
         Model_Name     MAE     MSE    RMSE      R2 Descriptors_Detail
0  All together r=9  0.5849  0.5752  0.7584  0.8621   All together r=9
